## COMPAS-Prediction
- what is COMPAS?
-- Correctional Offender Management Profiling for Alternative Sanctions.
- `It is a risk assessment system used in some U.S. courts.`
- The system predicts: 
        - `Will a person commit another crime (re-offend) within the next two years?`

In [1]:
import pandas as pd 
import numpy as np

c:\Users\iamak\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [7]:
df = pd.read_csv("compas-scores-two-years.csv")
df.shape
# df.columns

(7214, 53)

In [8]:
cols = [
    'age',
    'sex',
    'race',
    'priors_count',
    'juv_fel_count',
    'juv_misd_count',
    'juv_other_count',
    'c_charge_degree',
    'two_year_recid'
]

df = df[cols]

In [4]:
df=df.dropna()

In [5]:
df.head()

,age,sex,race,priors_count,juv_fel_count,juv_misd_count,juv_other_count,c_charge_degree,two_year_recid
0,69,Male,Other,0,0,0,0,F,0
1,34,Male,African-American,0,0,0,0,F,1
2,24,Male,African-American,4,0,0,1,F,1
3,23,Male,African-American,1,0,1,0,F,0
4,43,Male,Other,2,0,0,0,F,0


In [6]:
df.shape

(7214, 9)

In [9]:
y = df["two_year_recid"]

S = df[["sex","race"]]

X = df.drop(
    columns=[
        "two_year_recid",
        "sex",
        "race"
    ]
)

### Converting Dtype 

In [10]:
from sklearn.preprocessing import LabelEncoder

X_encoded = X.copy()

for col in X_encoded.columns:
    
    if X_encoded[col].dtype == "object" or X_encoded[col].dtype == "str":
        
        le = LabelEncoder()
        
        X_encoded[col] = le.fit_transform(
            X_encoded[col].astype(str)
        )

In [12]:
X_encoded.dtypes

age                int64
priors_count       int64
juv_fel_count      int64
juv_misd_count     int64
juv_other_count    int64
c_charge_degree    int64
dtype: object

In [11]:
S_encoded = S.copy()

for col in S_encoded.columns:

    le = LabelEncoder()

    S_encoded[col] = le.fit_transform(
        S_encoded[col].astype(str)
    )

In [13]:
S_encoded.dtypes

sex     int64
race    int64
dtype: object

### OB Functions

In [14]:
import numpy as np

def orthogonal_to_bias(X,S):

    X = np.asarray(X)

    S = np.asarray(S)

    beta = (
        np.linalg.pinv(
            S.T @ S
        )
        @ S.T
        @ X
    )

    X_fair = X - S @ beta

    return X_fair

In [15]:
X_fair = orthogonal_to_bias(
    X_encoded,
    S_encoded
)

### Traning/Testing Data

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

X_train,X_test,y_train,y_test = \
train_test_split(
    X_fair,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier()

model.fit(
    X_train,
    y_train
)

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

pred = xgb.predict(X_test)

acc = accuracy_score(
    y_test,
    pred
)

print(
    "Accuracy:",
    round(acc,4)
)

Accuracy: 0.6729


### SOB Algorithm

In [19]:
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler()
scaler_S = StandardScaler()

X_std = scaler_X.fit_transform(X_encoded)
S_std = scaler_S.fit_transform(S_encoded)

In [20]:
import numpy as np

beta = (
    np.linalg.pinv(S_std.T @ S_std)
    @ S_std.T
    @ X_std
)

X_ob = X_std - S_std @ beta

### Sparse Feature Selection (SOB Approximation)

In [21]:
from sklearn.linear_model import LogisticRegression

selector = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.05,
    max_iter=5000
)

selector.fit(X_ob, y)

c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l1'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",0.05
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass

In [22]:
important = np.abs(
    selector.coef_[0]
) > 1e-6

X_sob = X_ob[:, important]

print("Selected features:",
      X_sob.shape[1])

Selected features: 5


### Training Final Model 

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train,X_test,y_train,y_test = \
train_test_split(
    X_sob,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

xgb.fit(
    X_train,
    y_train
)

pred = xgb.predict(X_test)

acc = accuracy_score(
    y_test,
    pred
)

print(
    "Accuracy:",
    round(acc,4)
)

Accuracy: 0.6819


In [26]:
from sklearn.metrics import accuracy_score, roc_auc_score


In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, \
y_train, y_test, \
S_train, S_test = train_test_split(
    X_encoded,
    y,
    S_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## ML Baseline 

In [27]:
from sklearn.linear_model import LogisticRegression

X_ml = pd.concat(
    [X_encoded, S_encoded],
    axis=1
)

X_train_ml, X_test_ml, \
y_train_ml, y_test_ml, \
S_train_ml, S_test_ml = train_test_split(
    X_ml,
    y,
    S_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

ml_model = LogisticRegression(
    max_iter=5000
)

ml_model.fit(
    X_train_ml,
    y_train_ml
)

pred_ml = ml_model.predict(X_test_ml)

prob_ml = ml_model.predict_proba(
    X_test_ml
)[:,1]

In [28]:
ftu_model = LogisticRegression(
    max_iter=5000
)

ftu_model.fit(
    X_train,
    y_train
)

pred_ftu = ftu_model.predict(
    X_test
)

prob_ftu = ftu_model.predict_proba(
    X_test
)[:,1]


## SOB Transfromation

In [29]:
beta = (
    np.linalg.pinv(
        S_std.T @ S_std
    )
    @ S_std.T
    @ X_std
)

X_ob = X_std - S_std @ beta

In [30]:
selector = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.05,
    max_iter=5000
)

selector.fit(X_ob,y)

important = (
    np.abs(
        selector.coef_[0]
    ) > 1e-6
)

X_sob = X_ob[:,important]

c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


## SOB1

In [31]:
X_train_sob, X_test_sob, \
y_train_sob, y_test_sob, \
S_train_sob, S_test_sob = train_test_split(
    X_sob,
    y,
    S_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

sob1_model = LogisticRegression(
    max_iter=5000
)

sob1_model.fit(
    X_train_sob,
    y_train_sob
)

pred_sob1 = sob1_model.predict(
    X_test_sob
)

prob_sob1 = sob1_model.predict_proba(
    X_test_sob
)[:,1]

## SOB2 - Adding Sensitive Variable Back

In [32]:
X_sob2 = np.hstack([
    X_sob,
    S_encoded.values
])

X_train_sob2, X_test_sob2, \
y_train_sob2, y_test_sob2, \
S_train_sob2, S_test_sob2 = train_test_split(
    X_sob2,
    y,
    S_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

sob2_model = LogisticRegression(
    max_iter=5000
)

sob2_model.fit(
    X_train_sob2,
    y_train_sob2
)

pred_sob2 = sob2_model.predict(
    X_test_sob2
)

prob_sob2 = sob2_model.predict_proba(
    X_test_sob2
)[:,1]

In [33]:
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

def compute_metrics(
    y_true,
    pred,
    prob,
    sensitive
):

    y_true = np.asarray(y_true)
    pred = np.asarray(pred).astype(float)
    prob = np.asarray(prob).astype(float)
    sensitive = np.asarray(sensitive).astype(int)

    acc = accuracy_score(
        y_true,
        pred
    )

    auc = roc_auc_score(
        y_true,
        prob
    )

    groups = np.unique(sensitive)

    tprs = []

    for g in groups:

        mask = sensitive == g

        tp = np.sum(
            (y_true[mask] == 1) &
            (pred[mask] == 1)
        )

        fn = np.sum(
            (y_true[mask] == 1) &
            (pred[mask] == 0)
        )

        tprs.append(
            tp / (tp + fn + 1e-10)
        )

    eo = abs(
        tprs[0] - tprs[1]
    )

    rates = []

    for g in groups:

        mask = sensitive == g

        rates.append(
            float(
                np.mean(
                    pred[mask]
                )
            )
        )

    aa = abs(
        rates[0] - rates[1]
    )

    cf = abs(
        float(
            np.mean(
                prob[sensitive == 0]
            )
        )
        -
        float(
            np.mean(
                prob[sensitive == 1]
            )
        )
    )

    return [
        round(acc,4),
        round(auc,4),
        round(cf,4),
        round(eo,4),
        round(aa,4)
    ]

In [34]:
results = []

# ML
results.append(
    ["ML"] +
    compute_metrics(
        y_test_ml,
        pred_ml,
        prob_ml,
        S_test_ml["sex"]
    )
)

# FTU
results.append(
    ["FTU"] +
    compute_metrics(
        y_test,
        pred_ftu,
        prob_ftu,
        S_test["sex"]
    )
)

# SOB1
results.append(
    ["SOB1"] +
    compute_metrics(
        y_test_sob,
        pred_sob1,
        prob_sob1,
        S_test_sob["sex"]
    )
)

# SOB2
results.append(
    ["SOB2"] +
    compute_metrics(
        y_test_sob2,
        pred_sob2,
        prob_sob2,
        S_test_sob2["sex"]
    )
)

results_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "ACC",
        "AUC",
        "CF-Metric",
        "EO Fairness",
        "AA Fairness"
    ]
)

results_df

,Method,ACC,AUC,CF-Metric,EO Fairness,AA Fairness
0,ML,0.6826,0.7275,0.1315,0.3869,0.3205
1,FTU,0.6895,0.7327,0.0678,0.1722,0.1722
2,SOB1,0.6687,0.7157,0.0078,0.0105,0.0225
3,SOB2,0.6826,0.7276,0.1310,0.3869,0.3205


In [35]:
results = []

# ML
results.append(
    ["ML"] +
    compute_metrics(
        y_test_ml,
        pred_ml,
        prob_ml,
        S_test_ml["race"]
    )
)

# FTU
results.append(
    ["FTU"] +
    compute_metrics(
        y_test,
        pred_ftu,
        prob_ftu,
        S_test["race"]
    )
)

# SOB1
results.append(
    ["SOB1"] +
    compute_metrics(
        y_test_sob,
        pred_sob1,
        prob_sob1,
        S_test_sob["race"]
    )
)

# SOB2
results.append(
    ["SOB2"] +
    compute_metrics(
        y_test_sob2,
        pred_sob2,
        prob_sob2,
        S_test_sob2["race"]
    )
)

results_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "ACC",
        "AUC",
        "CF-Metric",
        "EO Fairness",
        "AA Fairness"
    ]
)

results_df

,Method,ACC,AUC,CF-Metric,EO Fairness,AA Fairness
0,ML,0.6826,0.7275,0.1124,0.2112,0.2758
1,FTU,0.6895,0.7327,0.0965,0.1417,0.1970
2,SOB1,0.6687,0.7157,0.0533,0.0027,0.0938
3,SOB2,0.6826,0.7276,0.1120,0.2112,0.2758


# Extension

In [36]:
import numpy as np

class AdaptiveFairnessState:
    def __init__(self, gamma=0.9):
        self.gamma = gamma
        self.state = None

    def initialize(self,S):
        self.state = S.copy()


    def update(self, fairness_signal):
        if fairness_signal.ndim == 1:
            fairness_signal = fairness_signal.reshape( 1,-1)
            

        # fairness_signal = np.repeat(
        #     fairness_signal,
        #     self.state.shape[0],
        #     axis=0
        # )

        # self.state = (
        #     self.gamma * self.state
        #     +
        #     (1-self.gamma) * fairness_signal
        # )

        fairness_signal = np.repeat(
            fairness_signal.reshape(1,-1),
            self.state.shape[0],
            axis=0
        )

        self.state = (
            self.gamma * self.state
            +
            (1-self.gamma) * fairness_signal
        )

    def get_state(self):
        return self.state

In [37]:
def compute_group_gap(preds, sensitive_column):

    group0 = preds[sensitive_column == 0]
    group1 = preds[sensitive_column == 1]

    if len(group0) == 0 or len(group1) == 0:
        return 0

    return group1.mean() - group0.mean()

In [38]:
def adaptive_ob(
        X,
        fairness_state):

    beta = (np.linalg.pinv(fairness_state.T@fairness_state)@fairness_state.T@X)

    X - fairness_state @ beta

In [41]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

afs = AdaptiveFairnessState(gamma=0.9)

afs.initialize( S_std)

T = 5

for t in range(T):
    F = afs.get_state()
    X_afs = adaptive_ob(X_std, F)

    def adaptive_ob(X, fairness_state):
        beta = np.linalg.pinv(fairness_state.T @ fairness_state) @ fairness_state.T @ X
        return X - fairness_state @ beta

    afs = AdaptiveFairnessState(gamma=0.9)
    afs.initialize(S_std)

    T = 5

    for t in range(T):
        F = afs.get_state()
        X_afs = adaptive_ob(X_std, F)

        X_train, X_test, y_train, y_test, S_train, S_test = train_test_split(
            X_afs,
            y,
            S_std,
            test_size=0.2,
            random_state=42
        )

    def adaptive_ob(X, fairness_state):

        beta = np.linalg.pinv(fairness_state.T @ fairness_state) @ fairness_state.T @ X

        return X - fairness_state @ beta

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42
    )

    model.fit(X_train, y_train)
    xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)
    xgb.fit(
    X_train,
    y_train)
    pred = xgb.predict(X_test)
    acc = accuracy_score(
    y_test,
    pred)
    gender_gap = compute_group_gap(pred, S_test[:,0])

    race_gap = compute_group_gap(pred, S_test[:,1])

    # fairness_signal = np.array([gender_gap, race_gap])

    fairness_signal = np.array([ abs(gender_gap), abs(race_gap)])
    afs.update(fairness_signal)

    print( f"Iteration {t+1}")
    print( "Accuracy:",round(acc,4))
    print("Gender Gap:",round(gender_gap,4))
    print( "Race Gap:",round(race_gap,4))
    print("-"*40)

Iteration 5
Accuracy: 0.6791
Gender Gap: 0
Race Gap: 0
----------------------------------------
Iteration 5
Accuracy: 0.6791
Gender Gap: 0
Race Gap: 0
----------------------------------------
Iteration 5
Accuracy: 0.6791
Gender Gap: 0
Race Gap: 0
----------------------------------------
Iteration 5
Accuracy: 0.6791
Gender Gap: 0
Race Gap: 0
----------------------------------------
Iteration 5
Accuracy: 0.6791
Gender Gap: 0
Race Gap: 0
----------------------------------------


In [42]:
F = afs.get_state()

X_afs = adaptive_ob(
    X_std,
    F
)

In [43]:
selector = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.05,
    max_iter=5000
)

selector.fit( X_afs,y)

important = (np.abs(selector.coef_[0]) > 1e-6)

X_sob_afs = X_afs[:, important]

c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\iamak\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [44]:
X_train,\
X_test,\
y_train,\
y_test = train_test_split(
    X_sob_afs,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit( X_train,y_train)
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42)
xgb.fit(
    X_train,
    y_train)
pred = xgb.predict(X_test)
prob = xgb.predict_proba(X_test)[:,1]

print( "Final Accuracy:",accuracy_score( y_test, pred))

Final Accuracy: 0.681912681912682


In [45]:
def adaptive_ob(X, fairness_state):

    beta = (
        np.linalg.pinv(
            fairness_state.T @ fairness_state
        )
        @ fairness_state.T
        @ X
    )

    X_fair = X - fairness_state @ beta

    return X_fair


In [46]:
X_train,\
X_test,\
y_train,\
y_test,\
S_train,\
S_test = train_test_split(
    X_sob_afs,
    y,
    S_encoded.values,
    test_size=0.2,
    random_state=42
)

## AUC 

In [47]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(
    y_test,
    prob
)

print("AUC:", round(auc,4))

AUC: 0.7225


### CF Metrics 
- Using Gender

In [48]:
gender = S_test[:,0]

cf_metric = abs(
    prob[gender==1].mean()
    -
    prob[gender==0].mean()
)

print(
    "CF Metric:",
    round(cf_metric,4)
)

CF Metric: 0.1288


## EO Fairness

In [49]:
def eo_fairness(
    y_true,
    y_pred,
    sensitive
):

    groups = np.unique(
        sensitive
    )

    tprs = []

    for g in groups:

        mask = sensitive == g

        tp = np.sum(
            (y_true[mask]==1)
            &
            (y_pred[mask]==1)
        )

        fn = np.sum(
            (y_true[mask]==1)
            &
            (y_pred[mask]==0)
        )

        tprs.append(
            tp/(tp+fn+1e-10)
        )

    return abs(
        tprs[0]-tprs[1]
    )

In [50]:
eo_gender = eo_fairness(
    y_test,
    pred,
    S_test[:,0]
)

eo_race = eo_fairness(
    y_test,
    pred,
    S_test[:,1]
)

### AA Fairness

In [51]:
def aa_fairness(
    pred,
    sensitive
):

    groups = np.unique(
        sensitive
    )

    rates = []

    for g in groups:

        mask = sensitive == g

        rates.append(
            pred[mask].mean()
        )

    return abs(
        rates[0]-rates[1]
    )

In [52]:
aa_gender = aa_fairness(
    pred,
    S_test[:,0]
)

aa_race = aa_fairness(
    pred,
    S_test[:,1]
)

##### Final AFS-SOB Metrics Table 

In [53]:
afs_results = pd.DataFrame({

    "Method":["AFS-SOB"],

    "ACC":[
        round(
            accuracy_score(
                y_test,
                pred
            ),
            4
        )
    ],

    "AUC":[
        round(
            auc,
            4
        )
    ],

    "CF-Metric":[
        round(
            cf_metric,
            4
        )
    ],

    "EO Fairness":[
        round(
            (eo_gender + eo_race)/2,
            4
        )
    ],

    "AA Fairness":[
        round(
            (aa_gender + aa_race)/2,
            4
        )
    ]
})

afs_results

,Method,ACC,AUC,CF-Metric,EO Fairness,AA Fairness
0,AFS-SOB,0.6819,0.7225,0.1288,0.4085,0.2298
